In [ ]:
#| default_exp j

# J
> Run the J language from Python and Jupyter through libj: sessions, magics and a Jupyter kernel.

`basedpl.j` runs the [J language](https://www.jsoftware.com/) inside the Python process. It loads libj, the engine library in every J installation, so no separate J process runs. The module provides the `J` session class, the `%%j` and `%j` magics, and a Jupyter kernel for J. Importing it does not start J.

Install J with `pip install jlanguage`, or from [jsoftware.com](https://www.jsoftware.com/). The Python layer needs `pip install 'basedpl[notebooks]'`.

In [ ]:
from fastcore.test import *

In [ ]:
#| export
import sys
from shutil import which
from fastcore.utils import *
from fastcore.xdg import xdg_config_home
from IPython.display import display
from basedpl._core import _J, JError, _install_kernelspec, _run_j_kernel
from basedpl.notebooks import AplOut

_all_ = ['JError']

## Finding J

In [ ]:
#| export
_LIBJ = 'j.dll' if sys.platform=='win32' else 'libj.dylib' if sys.platform=='darwin' else 'libj.so'

def find_j():
    "Locate the J binary directory: the one containing libj and profile.ijs"
    if (p:=which('jconsole')) and (d:=Path(p).resolve().parent/_LIBJ).exists(): return d.parent
    try:
        import jlang
        return Path(jlang.path())/'bin'
    except ImportError: pass
    roots = [Path.home(), Path('/Applications'), Path('/opt'), Path('/usr/share')]
    cands = sorted(c for r in roots if r.exists() for c in r.glob('j9*') if (c/'bin'/_LIBJ).exists())
    if cands: return cands[-1]/'bin'
    raise FileNotFoundError('J not found: pip install jlanguage, or install it from jsoftware.com')

`find_j` checks for libj beside `jconsole`, resolving symlinks first. The `jlanguage` package installs a `jconsole` script without libj beside it, so `find_j` asks the `jlang` module for that installation's path. It also checks common installation directories.

In [ ]:
jbin = find_j()
assert (jbin/_LIBJ).exists()
jbin.parts[-2:]

('jlang', 'bin')

## Sessions

In [ ]:
#| export
class J:
    "A J session: an engine from libj, with J's standard library loaded"
    def __init__(self, jbin=None): self._j = _J(Path(jbin or find_j())/_LIBJ)

`J` starts an engine, then runs `profile.ijs` to load J's standard library. Pass `jbin` to choose a J installation other than the one `find_j` returns.

In [ ]:
#| export
@patch
def run(self:J, code):
    "Run `code` (one or more lines of J), returning its output; raises `JError` on J errors"
    return self._j.run(code)

@patch
def __call__(self:J, code):
    "Run `code`, returning displayable output, or None if there is none"
    return AplOut(self.run(code)) or None

`run` returns J's output as text. Calling the session returns the same text as `AplOut`, which displays verbatim, or `None` when there is no output. Names persist between calls:

In [ ]:
j = J()
j('m =: 2 3 $ 10 * 1 + i. 6\nm')

10 20 30
40 50 60

In [ ]:
test_eq(j.run('+/ , m'), '210\n')
test_is(j('m2 =: 10 * m'), None)

Lines that open a multi-line definition take the following lines as its body:

In [ ]:
j('mean =: 3 : 0\n(+/ y) % # y\n)\nmean 1 2 3 4')

2.5

A J error raises `JError`. Its message is the session's output, including J's error display:

In [ ]:
with expect_fail(JError, contains='domain error'): j.run("m + 'x'")

Extended precision works, as in 25 factorial:

In [ ]:
j('*/ 1 + i. 25x')

15511210043330985984000000

## Python values

`getm` reads a named J noun as Python data. Characters become a string. Booleans, integers and floats become a number, or nested lists for an array. Boxed, extended and rational values raise `JError`.

In [ ]:
#| export
def _nest(x, shape):
    "Nest flat sequence `x` (row-major ravel) into `shape`"
    if len(shape)<2: return x
    n = len(x)//shape[0]
    return [_nest(x[i*n:(i+1)*n], shape[1:]) for i in range(shape[0])]

@patch
def getm(self:J, name):
    "Read noun `name` into Python: str for characters; int/float scalars and (nested) lists otherwise"
    shape,x = self._j.get(name)
    return _nest(x, shape) if shape or isinstance(x,str) else x[0]

In [ ]:
test_eq(j.getm('m'), [[10,20,30],[40,50,60]])
j('x =: 1r3')
with expect_fail(JError, contains='unsupported J type'): j.getm('x')

`j[expr]` evaluates an expression and returns its value as Python data. It assigns the result to `pytmp`, reads it, then erases that name.

In [ ]:
#| export
@patch
def pyval(self:J, expr):
    "Evaluate `expr` and return the result as a Python value"
    self.run(f'pytmp =: {expr}')
    try: return self.getm('pytmp')
    finally: self.run("4!:55 <'pytmp'")

@patch
def __getitem__(self:J, expr): return self.pyval(expr)

In [ ]:
test_eq(j['m > 25'], [[0,0,1],[1,1,1]])
test_eq(j['+/ % # 1 2 3 4'], 0.25)
j["'py' , 'val'"]

'pyval'

`j[name] = value` assigns Python data. A string becomes a J character list. A number or nested list becomes an integer array, or a float array if any item is a float.

In [ ]:
#| export
def _flat(o): return [a for x in o for a in _flat(x)] if isinstance(o,(list,tuple)) else [o]

@patch
def __setitem__(self:J, nm, v):
    "Assign Python value `v` (scalar, string, or nested list) to noun `nm`"
    if isinstance(v,str): return self._j.set(nm, [len(v.encode())], v)
    shape,x = [],v
    while isinstance(x,(list,tuple)): shape,x = shape+[len(x)],x[0]
    self._j.set(nm, shape, _flat(v))

In [ ]:
j['q'] = [[1,2],[3,4.5]]
j['s'] = "it's"
test_eq(j['s'], "it's")
j['+/ , q']

10.5

A ragged list has a different number of items than its shape implies, so assigning it raises `JError`:

In [ ]:
with expect_fail(JError, contains='shape'): j['r'] = [[1,2],[3]]

`fn` turns a J verb into a Python callable. Pass one argument for monadic use, or two for dyadic use with the left argument first.

In [ ]:
#| export
@patch
def fn(self:J, code):
    "A Python callable applying J verb `code` monadically or dyadically (left argument first)"
    def f(*args):
        if len(args)==1:
            self['pyy'] = args[0]
            return self.pyval(f'({code}) pyy')
        self['pyx'],self['pyy'] = args
        return self.pyval(f'pyx ({code}) pyy')
    return f

In [ ]:
sq = j.fn('*:')
test_eq(sq([1,2,3]), [1,4,9])
test_eq(j.fn('+/')([1,2,3]), 6)
j.fn('{.')(2, [5,6,7])

[5, 6]

## Interrupting

`interrupt` stops a running sentence with a J attention interrupt. Call it from another thread, because the thread running J cannot run Python code until J returns. The session stays usable after the interrupt.

In [ ]:
#| export
@patch
def interrupt(self:J):
    "Stop the running sentence with an attention interrupt; call it from another thread"
    self._j.interrupt()

In [ ]:
import threading, time
from concurrent.futures import ThreadPoolExecutor

In [ ]:
j('spin =: 3 : 0\nn =. 0\nwhile. n < 1e9 do. n =. n + 1 end.\n)')
t0 = time.time()
threading.Timer(0.5, j.interrupt).start()
with expect_fail(JError, contains='attention interrupt'): j('spin 0')
assert time.time()-t0 < 5
j('2+2')

4

`interrupt` is the only method that works from another thread. J sets its recursion limit from the stack of the thread that created the session. So a session runs J only on that thread, and raises `JError` on any other:

In [ ]:
with expect_fail(JError, contains='thread'): ThreadPoolExecutor().submit(j.run, '2+2').result()

## Exit requests and closing

`close` frees the engine. A `with` block calls `close` when it ends. A closed session cannot run code.

In [ ]:
#| export
@patch
def close(self:J):
    "Free the engine; the session cannot run code afterwards"
    self._j = None

@patch
def __enter__(self:J): return self

@patch
def __exit__(self:J, *args): self.close()

`exit 7` asks J to exit with code 7. The session records the code in `exited` and runs no more lines, without raising an error. The host decides what an exit request means.

In [ ]:
#| export
@patch(as_prop=True)
def exited(self:J):
    "The exit code requested by `exit`, or None"
    return self._j.exited

In [ ]:
with J() as j2:
    test_is(j2.exited, None)
    j2('exit 7')
    test_eq(j2.exited, 7)
    test_is(j2('2+2'), None)

## Magics

Run `%load_ext basedpl.j` to register two magics in IPython or Jupyter:

- `%%j` runs a cell and displays its output verbatim. A trailing `;` hides the output.
- `%j expr` returns the value of `expr` as Python data, as `j[expr]` does.

The magics share one session, which starts on the first call.

In [ ]:
#| export
class JMagic:
    "IPython `%j`/`%%j` magics, driving a lazily-started `J` session"
    def __init__(self, jbin=None): self.jbin,self.o = jbin,None

    def j(self, line, cell=None):
        "Run J: a cell magic displays the session output; a line magic returns the expression's Python value"
        if not self.o: self.o = J(self.jbin)
        if cell is None: return self.o[line.strip()]
        disp,cell = True,cell.rstrip()
        if cell.endswith(';'): disp,cell = False,cell[:-1]
        out = self.o(cell)
        if disp and out: display(out)

`load_ipython_extension` runs when IPython loads the extension. It calls `create_j_magic`, which registers a `JMagic` with any IPython shell:

In [ ]:
#| export
def create_j_magic(shell=None):
    "Create a `JMagic` and register its `j` line/cell magic with `shell`, returning it"
    if not shell: shell = get_ipython()
    jm = JMagic()
    shell.register_magic_function(jm.j, 'line_cell', 'j')
    return jm

def load_ipython_extension(ipython):
    "Register the `j` magics: `%load_ext basedpl.j`"
    create_j_magic(shell=ipython)

In [ ]:
%load_ext basedpl.j

In [ ]:
%%j
m3 =: 3 3 $ i. 9
m3 +/ . * m3

15 18  21
42 54  66
69 90 111

`%j` returns Python data, which you can assign:

In [ ]:
z = %j m3
z

[[0, 1, 2], [3, 4, 5], [6, 7, 8]]

A trailing `;` hides a cell's output:

In [ ]:
%%j
big =: 1000 1000 $ i. 5
big + big;

In [ ]:
#| hide
from IPython.utils.capture import capture_output

In [ ]:
#| hide
with capture_output() as cap: get_ipython().run_cell_magic('j', '', '2+2;')
test_eq(len(cap.outputs), 0)
with capture_output() as cap: get_ipython().run_cell_magic('j', '', '2+2')
test_eq(len(cap.outputs), 1)

## The J kernel

The J kernel runs J cells in Jupyter clients such as JupyterLab, nbclient, and agents using clikernel. Names persist across cells. An interrupt stops a computation and keeps the session. Install the kernel with `python -m basedpl.j install`. Jupyter then lists it as J.

In [ ]:
#| export
def install_j_kernel(
    prefix=None, # Install under `prefix/share/jupyter/kernels` if given, else in the user Jupyter directory
):
    "Register the J kernel with Jupyter as kernelspec `j`, returning its directory"
    return _install_kernelspec('j', [sys.executable, '-m', 'basedpl.j', '{connection_file}'], 'J', 'J', prefix)

`install_j_kernel` writes a kernelspec named `j`. It starts the kernel with `python -m basedpl.j CONNECTION_FILE`, using the current Python. Pass `prefix` to install into an environment, such as `sys.prefix`, instead of your user Jupyter directory.

`run_j_kernel` serves the kernel. It runs J on a Rust thread, through the same engine as `J`. Before the first request, it runs `basedpl/startup.ijs` from your XDG configuration directory if that file exists. The default path is `~/.config/basedpl/startup.ijs`.

In [ ]:
#| export
def run_j_kernel(connection_file):
    "Serve a J kernel on the Jupyter connection in `connection_file`"
    startup = xdg_config_home()/'basedpl'/'startup.ijs'
    _run_j_kernel(connection_file, Path(find_j())/_LIBJ, startup.read_text() if startup.exists() else None)

Running the module serves a kernel, or installs one when its argument is `install`:

In [ ]:
#| export
#| eval: false
if __name__ == '__main__':
    if sys.argv[1:] == ['install']: print(install_j_kernel())
    else: run_j_kernel(sys.argv[1])

## Limitations

- `%j` and `j[expr]` return booleans, integers, floats and characters as Python data. Boxed, extended and rational values raise `JError`. Use `%%j` to display them.
- macOS and Linux are supported. The libj binding is untested on Windows.

## Learning J

Start with [Learning J](https://www.jsoftware.com/help/learning/contents.htm) and the [J wiki](https://code.jsoftware.com/wiki/). Its [NuVoc](https://code.jsoftware.com/wiki/NuVoc) page is the reference for every primitive.